In [7]:
!pip3 install wget — quiet
!pip3 install openai==1.3.3 — quiet
!pip3 install sentence-transformers — quiet

ERROR: Invalid requirement: '—'
ERROR: Invalid requirement: '—'
ERROR: Invalid requirement: '—'


In [8]:
!pip3 install wget
!pip3 install openai==1.3.3
!pip3 install sentence-transformers

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9656 sha256=cdb1bce88859d064eebf5c9ceaa7ec7aee0ff81d9ce92dfaeadfcfb5371eabc4
  Stored in directory: /home/jovyan/.cache/pip/wheels/40/b3/0f/a40dbd1c6861731779f62cc4babcb234387e11d697df70ee97
Successfully built wget
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 43.7 MB/s eta 0:00:00
  Attempting uninstall: anyio
    Found existing installation: anyio 4.3.0
    Uninstalling anyio-4.3.0:
      Successfully uninstalled anyio-4.3.0
  Attempting uninstall: openai
    Found existing installation: openai 1.82.1
    Uninstalling openai-1.82.1:
      Successfully uninstalled openai-1.82.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821

In [11]:
import json
import os
import pandas as pd
import wget

In [19]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base')

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/383 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
cvs_file_path = 'https://raw.githubusercontent.com/openai/openai-cookbook/main/examples/data/AG_news_samples.csv'
file_path = 'AG_news_samples.csv'

if not os.path.exists(file_path):
    wget.download(cvs_file_path, file_path)
    print('File downloaded successfully.')
else:
    print('File already exists in the local file system.')

df = pd.read_csv('AG_news_samples.csv')
df

File downloaded successfully.


,title,description,label_int,label
0,World Briefings,BRITAIN: BLAIR WARNS OF CLIMATE THREAT Prime M...,1,World
1,Nvidia Puts a Firewall on a Motherboard (PC Wo...,PC World - Upcoming chip set will include buil...,4,Sci/Tech
2,"Olympic joy in Greek, Chinese press",Newspapers in Greece reflect a mixture of exhi...,2,Sports
3,U2 Can iPod with Pictures,"SAN JOSE, Calif. -- Apple Computer (Quote, Cha...",4,Sci/Tech
4,The Dream Factory,"Any product, any shape, any size -- manufactur...",4,Sci/Tech
...,...,...,...,...
1995,You Control: iTunes puts control in OS X menu ...,MacCentral - You Software Inc. announced on Tu...,4,Sci/Tech
1996,Argentina beat Italy for place in football final,Favourites Argentina beat Italy 3-0 this morni...,2,Sports
1997,NCAA case no worry for Spurrier,Shortly after Steve Spurrier arrived at Florid...,2,Sports
1998,Secret Service Busts Cyber Gangs,The US Secret Service Thursday announced arres...,4,Sci/Tech


In [16]:
data = df.to_dict(orient='records')
data[0]

{'title': 'World Briefings',
 'description': 'BRITAIN: BLAIR WARNS OF CLIMATE THREAT Prime Minister Tony Blair urged the international community to consider global warming a dire threat and agree on a plan of action to curb the  quot;alarming quot; growth of greenhouse gases.',
 'label_int': 1,
 'label': 'World'}

In [17]:
%%sql

DROP TABLE IF EXISTS news_articles;
CREATE TABLE IF NOT EXISTS news_articles (
    title TEXT,
    description TEXT,
    genre TEXT,
    embedding BLOB,
    FULLTEXT(title, description)
);

++
||
++
++

In [20]:
descriptions = [row['description'] for row in data]
all_embeddings = model.encode(descriptions)
all_embeddings.shape

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 768)

In [22]:
for row, embedding in zip(data, all_embeddings):
    row['embedding'] = embedding

In [23]:
data[0]

{'title': 'World Briefings',
 'description': 'BRITAIN: BLAIR WARNS OF CLIMATE THREAT Prime Minister Tony Blair urged the international community to consider global warming a dire threat and agree on a plan of action to curb the  quot;alarming quot; growth of greenhouse gases.',
 'label_int': 1,
 'label': 'World',
 'embedding': array([-1.42552713e-02, -1.03357071e-02,  1.25946105e-02,  8.40715785e-03,
        -6.92264410e-03, -8.77237227e-03, -5.38323671e-02,  1.95311196e-02,
         9.50564742e-02,  1.60899572e-02,  4.72200625e-02,  2.30231155e-02,
        -6.69442937e-02,  2.82599987e-03,  2.79738400e-02, -6.46088347e-02,
         5.52451760e-02, -4.02353071e-02, -2.22880822e-02, -1.65119395e-02,
         3.61824557e-02,  3.32110142e-03,  1.18329516e-02,  7.70277716e-03,
        -4.18954827e-02, -2.76368838e-02,  3.64982933e-02,  3.69321145e-02,
         5.97776957e-02,  8.05662386e-03,  3.38091105e-02, -1.52911590e-02,
         1.38111366e-02, -4.00905032e-03,  3.15332080e-08,  2.20

In [24]:
%sql TRUNCATE TABLE news_articles;

import sqlalchemy as sa
from singlestoredb import create_engine

# Use create_table from singlestoredb since it uses the notebook connection URL
conn = create_engine().connect()

statement = sa.text('''
    INSERT INTO news_articles (
        title,
        description,
        genre,
        embedding
    )
    VALUES (
        :title,
        :description,
        :label,
        :embedding
    )
''')

conn.execute(statement, data)

In [26]:
search_query = 'India'
search_embedding = model.encode(search_query)

query_statement = sa.text('''
    SELECT
        title,
        description,
        genre,
        DOT_PRODUCT(embedding, :embedding) AS score
    FROM news_articles
    ORDER BY score DESC
    LIMIT 10
''')

# Execute the SQL statement.
results = pd.DataFrame(conn.execute(query_statement, dict(embedding=search_embedding)))
print(results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

                                               title  \
0           Militants beat man thought to be from US   
1         Bomb at India Independence Parade Kills 15   
2        Microsoft Unveils Windows XP for India (AP)   
3  4 killed, 54 wounded in three separate attacks...   
4   Northeast Indian State Votes Amid Tight Security   
5  Why The Open-Source Model Can Work In India (T...   
6           Microsoft to Hire Hundreds More in India   
7  Bhopal victims commemorate 20th anniversary of...   
8  Follow-on shy Aussies lead India by 355 runs (...   
9                              No channel for series   

                                         description     genre     score  
0  HENDALA, Sri Lanka -- Day after day, locked in...     World  0.402497  
1  NEW DELHI - A bomb exploded during an Independ...     World  0.346617  
2  AP - Microsoft Corp. announced Wednesday that ...  Sci/Tech  0.344941  
3  Canadian Press - GAUHATI, India (AP) - Residen...     World  0.337517  
4   GUWA

In [27]:
hyb_query = 'Articles about India'
hyb_embedding = model.encode(hyb_query)

# Create the SQL statement.
hyb_statement = sa.text('''
    SELECT
        title,
        description,
        genre,
        DOT_PRODUCT(embedding, :embedding) AS semantic_score,
        MATCH(title, description) AGAINST (:query) AS keyword_score,
        (semantic_score + keyword_score) / 2 AS combined_score
    FROM news_articles
    ORDER BY combined_score DESC
    LIMIT 10
''')

# Execute the SQL statement.
hyb_results = pd.DataFrame(conn.execute(hyb_statement, dict(embedding=hyb_embedding, query=hyb_query)))
hyb_results

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,title,description,genre,semantic_score,keyword_score,combined_score
0,Why The Open-Source Model Can Work In India (T...,TechWeb - An Indian Institute of Technology pr...,Sci/Tech,0.326729,0.0,0.163365
1,Bomb at India Independence Parade Kills 15,NEW DELHI - A bomb exploded during an Independ...,World,0.324176,0.0,0.162088
2,"4 killed, 54 wounded in three separate attacks...","Canadian Press - GAUHATI, India (AP) - Residen...",World,0.316158,0.0,0.158079
3,Indian minister hits out at textbook praising ...,AFP - An Indian minister said a school text-bo...,World,0.282740,0.0,0.141370
4,PM inaugurates Guru Granth Sahib research cent...,Prime Minister Dr Manmohan Singh inaugurated a...,World,0.274033,0.0,0.137016
5,Turkish press jubilant over green light for EU...,"Adorned with Turkish and EU flags, Turkey #39;...",World,0.263389,0.0,0.131695
6,Northeast Indian State Votes Amid Tight Security,"GUWAHATI, India (Reuters) - People braved a s...",World,0.245400,0.0,0.122700
7,Putin favors veto right for India as permanent...,"In an apparent damage control exercise, Russia...",World,0.242365,0.0,0.121183
8,Blair Battles Own Political Party on UK Iraq P...,Description: NPR #39;s Alex Chadwick talks to ...,World,0.238868,0.0,0.119434
9,Search Engine Forums Spotlight,Links to this week's topics from search engine...,Sci/Tech,0.237496,0.0,0.118748
